# Net-Payout-Based Duration

## Step 0 — Setup, Imports, Pfade, Session


In [1]:
import re
import numpy as np
import pandas as pd
import lseg.data as ld
from pathlib import Path
import time
import random

import warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="lseg"
)

#Speicherstruktur für Intermediate und Final Output
BASE_DIR = Path(
    "/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data"
)
BASE_DIR.mkdir(parents=True, exist_ok=True)

(TABLE_DIR := BASE_DIR / "tables").mkdir(exist_ok=True)
(DATA_DIR  := BASE_DIR / "intermediate").mkdir(exist_ok=True)

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved: {path}")

def load_parquet(name: str) -> pd.DataFrame:
    path = DATA_DIR / f"{name}.parquet"
    return pd.read_parquet(path)

## Step 1 — Universum laden: STOXX 600 (EURO subset)

In [2]:
# --- User input ---
INPUT_XLSX = TABLE_DIR / "stoxx600_membership_matrix_1999_2025_eurohq.xlsx"
SHEET_NAME = 0  # or a sheet name string

df_const = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME)

# Robust column detection
ric_col = next(c for c in df_const.columns if "RIC" in c.upper())
name_col = next(c for c in df_const.columns if "COMPANY" in c.upper() or "NAME" in c.upper())
country_col = next(c for c in df_const.columns if "COUNTRY" in c.upper())

df_ric = df_const[[ric_col, name_col, country_col]].copy()
df_ric.columns = ["RIC", "CompanyName", "CountryHQ"]

df_ric = (
    df_ric
    .dropna(subset=["RIC"])
    .drop_duplicates(subset=["RIC"])
    .sort_values("RIC")
    .reset_index(drop=True)
)


## Step 2.1 — Market Value V_t

Funktion

In [7]:
def pull_market_cap_raw(
    rics,
    start="1999-01-01",
    end="2026-01-01",
):
    """
    Pull annual market cap exactly as delivered by LSEG.
    No aggregation, no transformation.
    Output columns:
        RIC | date | market_cap
    """
    df = ld.get_data(
        universe=list(rics),
        fields=[
            "TR.MarketCapLocalCurn.Date",
            "TR.MarketCapLocalCurn",
        ],
        parameters={
            "SDate": start,
            "EDate": end,
            "FRQ": "Y",
        },
    )

    # Make column names explicit and transparent
    df = df.rename(columns=lambda c: str(c).strip())

    # Expected columns: Instrument | Date | Company Market Cap
    df = df.rename(columns={
        "Instrument": "RIC",
        "Date": "date",
        "Company Market Cap": "market_cap",
    })

    # Parse types but do NOT alter dates
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["market_cap"] = pd.to_numeric(df["market_cap"], errors="coerce")

    return df[["RIC", "date", "market_cap"]].sort_values(["RIC", "date"]).reset_index(drop=True)

Pull Data

In [9]:
ld.open_session()

mktcap = pull_market_cap_raw(
    ["SAPG.DE"],  # ric_list
    start="1999-01-01",
    end="2026-01-01"
)

ld.close_session()

# Drop artificial last row with missing market cap
mktcap = mktcap.dropna(subset=["market_cap"]).reset_index(drop=True)
# Add calendar year explicitly
mktcap["year"] = mktcap["date"].dt.year

mktcap.head()

,RIC,date,market_cap,year
0,SAPG.DE,1999-12-30,29585000000.0,1999
1,SAPG.DE,2000-12-29,7459690000.0,2000
2,SAPG.DE,2001-12-28,26772900000.0,2001
3,SAPG.DE,2002-12-30,23775635731.199902,2002
4,SAPG.DE,2003-12-30,41937270988.900002,2003


# Step 2.2 Annual Dividends

In [11]:
# ============================================================
# Step 2.2 — Dividends (ANNUAL, raw & transparent)
# Field: TR.F.DivPaidCashTotCF  (Dividends Paid - Cash - Total - Cash Flow)
# Output columns: RIC | date | year | dividends
# Notes:
# - We request the companion .Date field to see the exact LSEG date
# - We drop rows where dividends are missing (often the artificial last row)
# ============================================================

def pull_dividends_raw(
    rics,
    start="1999-01-01",
    end="2026-01-01",
    field="TR.F.DivPaidCashTotCF",
):
    df = ld.get_data(
        universe=list(rics),
        fields=[f"{field}.Date", field],
        parameters={
            "SDate": start,
            "EDate": end,
            "FRQ": "Y",
        },
    )

    # keep names stable
    df = df.rename(columns=lambda c: str(c).strip())

    # expected: Instrument | Date | <pretty field name>
    df = df.rename(columns={
        "Instrument": "RIC",
        "Date": "date",
    })

    # value column = the one that's not RIC/date
    value_cols = [c for c in df.columns if c not in ["RIC", "date"]]
    if len(value_cols) != 1:
        raise ValueError(f"Ambiguous dividends column: {value_cols}. Columns: {list(df.columns)}")
    val_col = value_cols[0]

    df = df.rename(columns={val_col: "dividends"})

    # parse types (no transformation of dates beyond parsing)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["dividends"] = pd.to_numeric(df["dividends"], errors="coerce")

    # drop invalid rows (incl. artificial last NaN row)
    df = df.dropna(subset=["date"]).sort_values(["RIC", "date"]).reset_index(drop=True)
    df = df.dropna(subset=["dividends"]).reset_index(drop=True)

    # add explicit year
    df["year"] = df["date"].dt.year

    return df[["RIC", "date", "year", "dividends"]].sort_values(["RIC", "date"]).reset_index(drop=True)

In [12]:
ld.open_session()
div = pull_dividends_raw(
    ["SAPG.DE"],  # ric_list later
    start="1999-01-01",
    end="2026-01-01"
)
ld.close_session()
div = div[div["year"] >= 1999].reset_index(drop=True)
div.head()
div.tail()

,RIC,date,year,dividends
21,SAPG.DE,2020-12-31,2020,1864000000.0
22,SAPG.DE,2021-12-31,2021,2182000000.0
23,SAPG.DE,2022-12-31,2022,2865000000.0
24,SAPG.DE,2023-12-31,2023,2395000000.0
25,SAPG.DE,2024-12-31,2024,2565000000.0
